<a href="https://colab.research.google.com/github/ytlLab/python/blob/main/ch10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#多媒體圖片影片下載

## 下載Bing圖片

In [ ]:
!pip install bing-image-downloader==1.0.4 #匯入模組

In [ ]:
from bing_image_downloader import downloader #使用模組的downloader方法
#最多五張、儲存路徑bingimage資料夾、過濾成人圖片、強制覆寫圖片、最大連結時間20秒
downloader.download(
    "WBC",
    limit = 5,
    output_dir = 'bingimage',
    adult_filter_off = True,
    force_replace = True,
    timeout = 20
)

##下載google圖片

###Google Custom Search API

Google 官方提供 API 供開發者搜尋圖片，每天前 100 次搜尋免費。


1.到Google Cloud Console 建立專案並取得 API Key。

2.到Custom Search Engine 建立一個搜尋引擎，開啟「圖片搜尋」並取得 CX ID。



In [ ]:
import requests
import os

def google_search(search_term, api_key, cse_id, num=10):
    url = f"https://www.googleapis.com/customsearch/v1"
    params = {
        'q': search_term,
        'key': api_key,
        'cx': cse_id,
        'searchType': 'image',
        'num': num  # 每次最多 10 張
    }
    response = requests.get(url, params=params)
    return response.json()

# 使用範例
API_KEY = "你的_API_KEY"
CX_ID = "你的_CX_ID"
results = google_search("台北美食", API_KEY, CX_ID)

for i, item in enumerate(results.get('items', [])):
    img_url = item['link']
    print(f"找到圖片 {i+1}: {img_url}")
    # 接下來用 requests.get(img_url) 下載即可

###google-search-results (SerpApi)


一個商業 API，它專門幫你處理「爬蟲被擋」的問題。它會回傳乾淨的 JSON 資料，你完全不需要寫解析 HTML 的程式碼。

優點：極其穩定，不用維護 Selenium。

缺點：每月有免費額度限制（約 100 次搜尋）。

In [ ]:
!pip install google-search-results

In [2]:
import os
import requests
from serpapi import GoogleSearch

In [ ]:
# --- 設定區域 ---
SERPAPI_KEY = "你的_API_KEY" # 請填入你的 Key
keyword = "台灣黑熊"
num_images = 20 # 想要下載的張數
save_dir = "serp_images"

if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# --- 執行搜尋 ---
params = {
    "engine": "google_images", # 指定搜尋引擎為 Google Images
    "q": keyword,
    "api_key": SERPAPI_KEY
}

search = GoogleSearch(params)
results = search.get_dict()

# 檢查是否有回傳圖片結果
if "images_results" in results:
    images = results["images_results"]

    count = 0
    for i, img in enumerate(images):
        if count >= num_images:
            break

        img_url = img.get("original") # 這裡可以拿 'original' 原圖或 'thumbnail' 縮圖
        if not img_url: continue

        try:
            print(f"正在下載第 {count+1} 張: {img_url[:60]}...")
            # 下載圖片
            response = requests.get(img_url, timeout=10)
            if response.status_code == 200:
                # 取得副檔名 (簡單判斷)
                ext = img_url.split('.')[-1].split('?')[0]
                if ext.lower() not in ['jpg', 'jpeg', 'png', 'gif']:
                    ext = 'jpg'

                with open(f"{save_dir}/image_{count+1}.{ext}", "wb") as f:
                    f.write(response.content)
                count += 1
        except Exception as e:
            print(f"⚠️ 下載失敗: {e}")

    print(f"\n✅ 下載完成！共存入 {count} 張圖片。")
else:
    print("❌ 找不到圖片結果，請檢查關鍵字或 API Key。")


##其他(duckduckgo)

duckduckgo_search圖片爬蟲套件。能抓取 DuckDuckGo 的搜尋結果（背後整合了多個引擎，包含 Bing）。

In [ ]:
!pip install duckduckgo_search

In [ ]:
from duckduckgo_search import DDGS
import os
import requests

In [ ]:
save_dir = "ddg_images"
os.makedirs(save_dir, exist_ok=True)

with DDGS() as ddgs:
    # images() 會回傳一個產生器
    keywords = "玉山 日出"
    results = ddgs.images(keywords, max_results=20)

    for i, r in enumerate(results):
        print(f"正在下載第 {i+1} 張: {r['image']}")
        try:
            res = requests.get(r['image'], timeout=10)
            with open(f"{save_dir}/{i}.jpg", "wb") as f:
                f.write(res.content)
        except:
            print("下載失敗，跳過")

print("✅ 下載完成")

##下載Youtube影片

In [ ]:
!pip install pytubefix

In [10]:
from pytubefix import YouTube
from pytubefix.cli import on_progress

In [ ]:
url = 'https://www.youtube.com/watch?v=27ob2G3GUCQ'

yt = YouTube(url, on_progress_callback = on_progress)
print(yt.title)

In [ ]:
print(yt.streams) #取得影片格式

In [ ]:
yt.streams.first().download("youtube") #下載影片 first()傳回第一個影片格式

In [ ]:
print(len(yt.streams.filter(adaptive=True))) #filter()傳回符合指定條件的影片格式

In [ ]:
print(yt.streams.filter(progressive=True))
#progressive 篩選同時具有影像及聲音的格式
#adaptive 篩選只具有影像或聲音其中一種格式
#subtype 篩選指定影片類型的格式 如subtype='mp4'
#res 篩選指定解析度的格式 res='720'

In [ ]:
yt.streams.filter(subtype='mp4', res='360p', progressive=True).first().download("youtube")

#作業1
銜接上學期matlab類神經網路課程，將圖片蒐集的步驟，
改用python圖片下載的方式呈現，並將圖片mount到自己的雲端硬碟。

#作業2

爬取youtube資料

輸入關鍵字搜尋，
提供相關縮圖、標題、連結

#補充 Try-Except

Python 例外處理：Try-Except

在編寫程式時，難免會遇到錯誤（如除以零、讀取不存在的檔案）。

為了防止程式直接崩潰（Crash），使用 Try-Except 機制來捕捉並處理這些「例外」。

核心語法結構
* try：放置可能產生錯誤的程式碼。
* except：當錯誤發生時，執行的應變措施。
* else：如果 try 區塊順利執行完畢（無錯誤），則執行此處。
* finally：無論是否發生錯誤，最後都一定會執行的清理工作（如關閉檔案）。

優點
1. 提高穩定性：避免程式因為小錯誤而整個中斷。
2. 友善提示：提供具體的錯誤訊息，而不是讓使用者看到看不懂的系統報錯（Traceback）。
3. 資源管理：確保即使發生錯誤，網路連線或檔案也能透過 finally 被正確關閉。




In [ ]:
try:
    num = int(input("請輸入一個數字: "))
    result = 10 / num
except ZeroDivisionError:
    print("錯誤：除數不能為零！")
except ValueError:
    print("錯誤：請輸入有效的整數！")
else:
    print(f"計算成功，結果為: {result}")
finally:
    print("程式執行結束。")

字串格式化方式

Python 3.6 版本導入f-string (Formatted String Literals)語法
三個核心部分：

* f前綴：在字串引號"前加上f或F，告訴Python這是一個「格式化字串」。
* 字串內容："你輸入的數字是 ..."，這是你想顯示的文字。
* 大括號 {}：這是預留位置(Placeholder)。Python會執行大括號內的運算式，並將結果轉換為字串填入該位置。


In [ ]:
#捕捉特定錯誤 (ValueError)
try:
    num = int(input("請輸入一個數字: "))
    print(f"你輸入的數字是 {num}")
except ValueError:
    print("錯誤：這不是一個有效的整數！")

In [ ]:
#捕捉除以零錯誤 (ZeroDivisionError)
try:
    result = 10 / 0
except ZeroDivisionError:
    print("錯誤：數學運算中除數不能為零。")

In [ ]:
#使用多個 except 區塊來處理不同的錯誤情境。
try:
    list_a = [1, 2, 3]
    print(list_a[5])  # 索引越界
except IndexError:
    print("錯誤：索引超出範圍。")
except Exception as e:
    print(f"發生了其他錯誤：{e}")

In [ ]:
#搭配 else 使用
#else 區塊只有在 try 裡面沒有任何錯誤時才會執行。
try:
    f = open("test.txt", "w")
    f.write("Hello Python")
except IOError:
    print("錯誤：無法寫入檔案。")
else:
    print("檔案寫入成功！")
    f.close()

In [ ]:
#搭配 finally 使用（無論如何都執行）
#常用於釋放資源，例如關閉資料庫連線或檔案，確保程式清理工作必定完成。
try:
    x = 1 / 1
except ZeroDivisionError:
    print("發生除零錯誤")
finally:
    print("無論是否發生錯誤，這行都會執行。")

自定義except範例

Python內建的錯誤（如ValueError）很通用，如果需要一個更明確的理由。

例如：你想規定「存款金額不能是負數」，這在數學上沒錯，但在你的銀行系統裡是錯的。

建立一個新的類別（Class），並讓它繼承 Python 原有的 Exception 類別即可。

In [ ]:
#這個範例模擬一個銀行存款功能，如果金額小於 0，就丟出我們自定義的錯誤。
# 1. 定義你的專屬錯誤標籤
class MoneyError(Exception):
    """當金額為負數時觸發的錯誤"""
    pass

# 2. 使用這個錯誤
def deposit(amount):
    if amount < 0:
        # 主動丟出這個錯誤
        raise MoneyError(f"錯誤！你不能存入負數金額：{amount}")
    else:
        print(f"成功存入 {amount} 元！")

# 3. 捕捉這個錯誤
try:
    user_input = -100
    deposit(user_input)
except MoneyError as e:
    # 這裡只會捕捉到我們自定義的 MoneyError
    print(f"通知經理：{e}")

raise -> 丟出錯誤

在程式執行過程中，Python 官方定義了一些標準錯誤（如除以零）。

但有時候，雖然程式碼語法正確，但邏輯上不允許它發生，這時候就要用 raise 主動叫程式「停下來，這是不對的！」。

In [ ]:
#基本用法（主動丟出內建錯誤）
#當條件不符合時，強制程式中斷並報錯。

age = -5
if age < 0:
    raise ValueError("年齡不能是負數！")

In [ ]:
#在自定義功能中使用
#讓呼叫這個功能的人知道他們傳進了錯誤的資料。
def set_password(pwd):
    if len(pwd) < 6:
        raise Exception("密碼太短了，至少要 6 個字！")
    print("密碼設定成功")

set_password("123")

In [ ]:
#搭配 try...except（轉發錯誤）
#有時候抓到錯誤，處理完一部分（例如寫個紀錄），再把錯誤丟出去給別人處理。
try:
    num = 10 / 0
except ZeroDivisionError:
    print("記錄錯誤：有人嘗試除以零")
    raise  # 不帶參數的 raise 會把剛才抓到的錯誤再次丟出去

In [ ]:
#丟出「自定義」的標籤
#搭配我們上次說的自定義錯誤類別。
class OutOfFuel(Exception):
    pass
fuel = 0
if fuel == 0:
    raise OutOfFuel("油箱空了，車子無法啟動！")

In [ ]:
#限制輸入類型
#確保使用者傳進來的是正確的資料格式。
name = 12345
if not isinstance(name, str):
    raise TypeError("姓名必須是字串格式！")